# Filter raw expression data by log2 thresholds

This notebook converts the `filter_raw_data.py` script into modular, well-documented steps. Each function is in its own cell and explained.

Goals:
- Read a tab-separated file with encoding fallbacks (to avoid UnicodeDecodeError)
- Detect SampleA/SampleB columns flexibly
- Compute log2 values and apply thresholds
- Write filtered results to `filtered_data.txt`

Dataset path (relative to this notebook): `raw_data.txt`

In [1]:
# Imports used across functions
import csv, math, os, sys
from typing import List, Tuple

# Notebook utility: show versions used
print(f"Python version: {sys.version.split()[0]}")

Python version: 3.12.2


## 1) Reading text with encoding fallbacks

Text files from various sources may not be UTF-8. We'll read with a few common encodings and pick the first that works.

In [2]:
from typing import List, Tuple

def read_text_with_fallbacks(path: str) -> Tuple[List[str], str]:
    """Read text file lines trying multiple encodings. Returns (lines, encoding).

    Tries: utf-8, utf-8-sig, latin-1.
    """
    encodings = ["utf-8", "utf-8-sig", "latin-1"]
    last_err: Exception | None = None
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.readlines(), enc
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Failed to read '{path}' with encodings {encodings}: {last_err}")

## 2) Normalizing header names

We standardize column names by lowercasing, trimming, and replacing underscores with spaces to improve matching.

In [3]:
def normalize_header_name(name: str) -> str:
    n = name.strip().lower().replace("_", " ")
    return " ".join(n.split())  # collapse duplicate spaces

## 3) Locating SampleA and SampleB columns

We support common variants like “SampleA”, “Sample A”, and “Sample A data”. If not found, a fallback tries exact labels; if still not found, the caller may choose to assume the last two columns.

In [4]:
from typing import List, Tuple

def find_sample_columns(header: List[str]) -> Tuple[int, int]:
    normalized = [normalize_header_name(h) for h in header]

    candidates_a = {
        "samplea",
        "sample a",
        "sample a data",
        "sample a value",
        "sample a values",
    }
    candidates_b = {
        "sampleb",
        "sample b",
        "sample b data",
        "sample b value",
        "sample b values",
    }

    idx_a = idx_b = -1
    for i, n in enumerate(normalized):
        if n in candidates_a and idx_a == -1:
            idx_a = i
        if n in candidates_b and idx_b == -1:
            idx_b = i

    if idx_a == -1 or idx_b == -1:
        try:
            idx_a = header.index("SampleA") if idx_a == -1 else idx_a
        except ValueError:
            pass
        try:
            idx_b = header.index("SampleB") if idx_b == -1 else idx_b
        except ValueError:
            pass

    if idx_a == -1 or idx_b == -1:
        raise ValueError(
            f"Could not find both SampleA and SampleB columns in header: {header}"
        )
    return idx_a, idx_b

## 4) Parsing floats safely

We’ll turn a string into a float or return `None` if it’s not valid.

In [5]:
def parse_float_safe(value: str):
    try:
        return float(value)
    except Exception:
        return None

## 5) Row-level filtering logic

We compute log2 for SampleA and SampleB (only for positive values). A row passes if `max(log2A, log2B) > 2` and `|log2A − log2B| > 3`.

In [6]:
def row_passes(a: float, b: float) -> bool:
    if a <= 0 or b <= 0:
        return False
    log2a = math.log2(a)
    log2b = math.log2(b)
    cond1 = max(log2a, log2b) > 2
    cond2 = abs(log2a - log2b) > 3
    return cond1 and cond2

## 6) Putting it together: filter_file

This function reads the file, detects the SampleA/B columns, applies the filter, and writes the output TSV with the header preserved.

In [7]:
def filter_file(input_path: str, output_path: str) -> Tuple[int, int]:
    text_lines, used_encoding = read_text_with_fallbacks(input_path)

    reader = csv.reader(text_lines, delimiter="\t")
    rows = list(reader)
    if not rows:
        raise ValueError(f"Input file '{input_path}' appears to be empty.")

    header = rows[0]
    if len(header) < 2:
        raise ValueError(
            f"Header seems malformed (fewer than 2 columns): {header} from file {input_path}"
        )
    try:
        idx_a, idx_b = find_sample_columns(header)
    except ValueError as e:
        if len(header) >= 2:
            idx_a, idx_b = len(header) - 2, len(header) - 1
        else:
            raise e

    kept: List[List[str]] = [header]
    total = 0
    kept_count = 0
    for row in rows[1:]:
        if not row or all(c.strip() == "" for c in row):
            continue
        total += 1
        if len(row) <= max(idx_a, idx_b):
            continue
        a_val = parse_float_safe(row[idx_a])
        b_val = parse_float_safe(row[idx_b])
        if a_val is None or b_val is None:
            continue
        if row_passes(a_val, b_val):
            kept.append(row)
            kept_count += 1

    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    with open(output_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerows(kept)

    return total, kept_count

## 7) Optional CLI wrapper (for completeness)

The original script also included a `main` function to allow command-line usage. We keep it here but won’t run it automatically in the notebook.

In [8]:
def main(argv: List[str]) -> int:
    script_dir = os.path.dirname(os.path.abspath("."))
    default_in = os.path.join(script_dir, "raw_data.txt")
    default_out = os.path.join(script_dir, "filtered_data.txt")

    input_path = argv[1] if len(argv) > 1 else default_in
    output_path = argv[2] if len(argv) > 2 else default_out

    try:
        total, kept = filter_file(input_path, output_path)
    except Exception as e:
        print(f"Error: {e}")
        return 1

    print(
        f"Read {total} data rows from '{input_path}' and kept {kept} rows. Output -> '{output_path}'"
    )
    return 0

## 8) Quick demo: run on `raw_data.txt`

This will read `raw_data.txt` in the same folder as the notebook and write `filtered_data.txt`. We’ll print a brief summary.

In [9]:
# Demo cell: run the filter
in_path = os.path.join(os.path.dirname("."), "raw_data.txt")
out_path = os.path.join(os.path.dirname("."), "filtered_data.txt")
try:
    total, kept = filter_file(in_path, out_path)
    print(f"Read {total} data rows and kept {kept}. Output -> {out_path}")
except FileNotFoundError:
    print("raw_data.txt not found next to this notebook. Adjust paths if needed.")

Read 173869 data rows and kept 9830. Output -> filtered_data.txt
